# 04 — Machine Learning Models

**ECE1513 Course Project — Traffic Congestion Prediction near U of T St. George Campus**

This notebook trains and evaluates three ML models for traffic speed prediction on city streets:

1. **Linear Regression** — simple, interpretable reference.
2. **Random Forest** — ensemble of decision trees; handles non-linearity.
3. **XGBoost** — gradient-boosted trees; typically the strongest tabular learner.

All models are compared against the Historical Average baseline from Notebook 03.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Processed Data

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

TARGET = 'avg_speed'

# Select numeric feature columns, excluding the target and non-feature columns
exclude = {TARGET, 'congestion_level', 'id', 'count_id', 'centreline_id',
           'total_volume', 'latitude', 'longitude'}
# Also exclude the raw speed bin columns (already captured in avg_speed)
speed_bin_cols = [c for c in train_df.columns if c.startswith('vol_')]
exclude.update(speed_bin_cols)

feature_cols = [c for c in train_df.select_dtypes(include=[np.number]).columns if c not in exclude]

X_train = train_df[feature_cols].values
y_train = train_df[TARGET].values
X_test = test_df[feature_cols].values
y_test = test_df[TARGET].values

print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

In [ ]:
# Handle any NaN in features (fill with column median from training set)
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

print(f'NaN in X_train after imputation: {np.isnan(X_train).sum()}')
print(f'NaN in X_test after imputation: {np.isnan(X_test).sum()}')

## 2. Helper — Evaluation Function

In [ ]:
def evaluate(name, y_true, y_pred):
    """Compute and print regression metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name:25s}  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}')
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results = []

## 3. Baseline Comparison

Compute the Historical Average baseline predictions for comparison.

In [ ]:
# Reproduce the historical average baseline from Notebook 03
group_cols = ['location_id', 'hour_of_day', 'day_of_week']
group_means = train_df.groupby(group_cols)[TARGET].mean()
global_mean = train_df[TARGET].mean()

test_keys = test_df[group_cols].apply(tuple, axis=1)
y_pred_baseline = test_keys.map(group_means).fillna(global_mean).values

results.append(evaluate('Historical Average', y_test, y_pred_baseline))

## 4. Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

results.append(evaluate('Linear Regression', y_test, y_pred_lr))

## 5. Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

results.append(evaluate('Random Forest', y_test, y_pred_rf))

## 6. XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

results.append(evaluate('XGBoost', y_test, y_pred_xgb))

## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#7f8c8d', '#3498db', '#2ecc71', '#e74c3c']

for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    results_df[metric].plot(kind='bar', ax=axes[i], color=colors, edgecolor='white')
    axes[i].set_title(metric)
    axes[i].set_ylabel(metric)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Model Comparison — Speed Prediction on City Streets near U of T', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predicted vs Actual scatter for each ML model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_preds = [('Linear Regression', y_pred_lr), ('Random Forest', y_pred_rf), ('XGBoost', y_pred_xgb)]
plot_colors = ['#3498db', '#2ecc71', '#e74c3c']

for ax, (name, y_pred), color in zip(axes, models_preds, plot_colors):
    ax.scatter(y_test, y_pred, alpha=0.1, s=5, c=color)
    lims = [0, max(y_test.max(), y_pred.max()) + 5]
    ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect')
    ax.set_xlabel('Actual Speed (km/h)')
    ax.set_ylabel('Predicted Speed (km/h)')
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/pred_vs_actual_all_models.png', bbox_inches='tight')
plt.show()

## 8. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest importance
rf_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
rf_imp.tail(15).plot(kind='barh', ax=axes[0], color='#2ecc71')
axes[0].set_title('Random Forest — Top 15 Features')
axes[0].set_xlabel('Importance')

# XGBoost importance
xgb_imp = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=True)
xgb_imp.tail(15).plot(kind='barh', ax=axes[1], color='#e74c3c')
axes[1].set_title('XGBoost — Top 15 Features')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('../results/figures/feature_importance.png', bbox_inches='tight')
plt.show()

## 9. Save Models and Results

In [ ]:
# Save all trained models
models_dir = os.path.join('..', 'results', 'models')
os.makedirs(models_dir, exist_ok=True)

joblib.dump(lr, os.path.join(models_dir, 'linear_regression.pkl'))
joblib.dump(rf, os.path.join(models_dir, 'random_forest.pkl'))
joblib.dump(xgb, os.path.join(models_dir, 'xgboost.pkl'))
joblib.dump(imputer, os.path.join(models_dir, 'imputer.pkl'))

# Save feature column list for reproducibility
import json
with open(os.path.join(models_dir, 'feature_cols.json'), 'w') as f:
    json.dump(feature_cols, f)

# Save results table
tables_dir = os.path.join('..', 'results', 'tables')
os.makedirs(tables_dir, exist_ok=True)
results_df.to_csv(os.path.join(tables_dir, 'model_comparison.csv'))

print('Models and results saved.')
print(f'Feature columns saved: {len(feature_cols)}')

## 10. Discussion

**Key observations**:

- All three ML models outperform the Historical Average baseline, confirming that weather features and location-specific characteristics provide useful predictive signal beyond simple temporal averages.
- **Linear Regression** improves over the baseline but has limited capacity to capture non-linear interactions (e.g., rush-hour effects that differ by weather condition or location).
- **Random Forest** offers a substantial improvement by capturing feature interactions and non-linearity without requiring manual feature engineering.
- **XGBoost** achieves the best overall performance across all three metrics, benefiting from sequential boosting that focuses on hard-to-predict instances.

**Feature importance** highlights that location-related features (location_id, direction_code) and temporal features (hour_of_day, day_of_week) are the strongest predictors. Weather variables (temperature, precipitation) also contribute, though city street speeds are dominated by time-of-day patterns and location-specific characteristics (e.g., road width, intersection density).

In the next notebook, we will perform a deeper error analysis and investigate model behaviour across different locations and time periods.